# Notebook 6 — A/B Test Analysis

**Business scenario:** The retention team wants to test whether offering a 10%
monthly discount to high-risk customers reduces churn. They ran an experiment
for 30 days, splitting 1,000 high-risk customers into two groups: a control
group (no discount) and a treatment group (10% discount offered).

**What this notebook does:** Simulates the experiment, runs a pre-registered
power analysis, sanity-checks the experiment design, tests the result for
statistical and practical significance, breaks results down by customer
segment, and produces a decision memo recommending GO / NO-GO.

**Input:** None (the experiment is simulated with a fixed random seed)

**Output:**
- `data/powerbi/ab_test_results.csv`
- Charts saved to `outputs/charts/ab_test/`

**Estimated run time:** ~30 seconds

In [1]:
# Imports and shared project utilities
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, proportion_confint
from IPython.display import Markdown, display

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import set_style, save_chart, save_dataframe, project_path

set_style()

## Simulate the experiment

In [2]:
# All random draws use the same RandomState(42) so the experiment is fully reproducible,
# drawn in a fixed order: control churn, control charges, treatment churn, treatment
# charges, then tenure segments for the combined sample.
rng = np.random.RandomState(42)

control_churned = rng.binomial(1, 0.35, 500)
control_charges = np.round(rng.uniform(50, 110, 500), 2)

treatment_churned = rng.binomial(1, 0.26, 500)
treatment_charges = np.round(rng.uniform(50, 110, 500), 2)

tenure_segment_labels = ["Short <12m", "Medium 12-36m", "Long >36m"]
tenure_segments = rng.choice(tenure_segment_labels, size=1000, p=[0.4, 0.35, 0.25])

ab_data = pd.DataFrame({
    "customer_id": [f"C_{i:04d}" for i in range(1, 501)] + [f"T_{i:04d}" for i in range(1, 501)],
    "group": ["control"] * 500 + ["treatment"] * 500,
    "churned": np.concatenate([control_churned, treatment_churned]),
    "monthly_charges": np.concatenate([control_charges, treatment_charges]),
    "tenure_segment": tenure_segments,
})

print("Experiment sample:")
print(ab_data["group"].value_counts())
ab_data.head()

Experiment sample:
group
control      500
treatment    500
Name: count, dtype: int64


,customer_id,group,churned,monthly_charges,tenure_segment
0,C_0001,control,0,91.89,Short <12m
1,C_0002,control,1,82.17,Short <12m
2,C_0003,control,1,68.57,Long >36m
3,C_0004,control,0,98.83,Short <12m
4,C_0005,control,0,91.08,Short <12m


## Step 1 — Power analysis (run before looking at results)

In [3]:
# Minimum detectable effect: reducing churn from 35% (baseline) to 30% (5pp absolute reduction)
baseline_rate = 0.35
minimum_detectable_rate = 0.30

effect_size = proportion_effectsize(baseline_rate, minimum_detectable_rate)
required_n_per_group = zt_ind_solve_power(
    effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0, alternative="two-sided"
)

print(f"Baseline churn rate: {baseline_rate:.0%}")
print(f"Minimum detectable effect: {minimum_detectable_rate:.0%} (5pp absolute reduction)")
print(f"Effect size (Cohen's h): {effect_size:.4f}")
print(f"Required sample size per group (alpha=0.05, power=0.80): {required_n_per_group:.0f}")
print(f"Actual sample size per group: 500")

if required_n_per_group <= 500:
    print("\nOur 500 customers per group IS sufficient to detect a 5pp reduction at 80% power.")
else:
    print(f"\nOur 500 customers per group is NOT sufficient — the study is underpowered by "
          f"{required_n_per_group - 500:.0f} customers per group to reliably detect a 5pp reduction "
          f"at 80% power. Any significant result found below should be treated with caution and "
          f"the test should ideally be re-run with a larger sample or longer duration.")

Baseline churn rate: 35%
Minimum detectable effect: 30% (5pp absolute reduction)
Effect size (Cohen's h): 0.1068
Required sample size per group (alpha=0.05, power=0.80): 1376
Actual sample size per group: 500

Our 500 customers per group is NOT sufficient — the study is underpowered by 876 customers per group to reliably detect a 5pp reduction at 80% power. Any significant result found below should be treated with caution and the test should ideally be re-run with a larger sample or longer duration.


## Step 2 — Sanity checks

In [4]:
# Check 1: Sample Ratio Mismatch (SRM) — are both groups exactly the expected size?
group_sizes = ab_data["group"].value_counts()
control_n = group_sizes["control"]
treatment_n = group_sizes["treatment"]

print(f"Control group size: {control_n}")
print(f"Treatment group size: {treatment_n}")
print(f"Both groups exactly 500: {control_n == 500 and treatment_n == 500}")

chi2_stat, chi2_p = stats.chisquare([control_n, treatment_n], f_exp=[500, 500])
srm_result = "PASS" if chi2_p > 0.01 else "FAIL"
print(f"Chi-square test on group sizes: statistic={chi2_stat:.4f}, p-value={chi2_p:.4f}")
print(f"Sample Ratio Mismatch check: {srm_result}")

Control group size: 500
Treatment group size: 500
Both groups exactly 500: True
Chi-square test on group sizes: statistic=0.0000, p-value=1.0000
Sample Ratio Mismatch check: PASS


In [5]:
# Check 2: Pre-experiment balance — are the groups comparable on monthly_charges?
control_charges_series = ab_data.loc[ab_data["group"] == "control", "monthly_charges"]
treatment_charges_series = ab_data.loc[ab_data["group"] == "treatment", "monthly_charges"]

t_stat, t_p = stats.ttest_ind(control_charges_series, treatment_charges_series)
balance_result = "groups are balanced (p > 0.05)" if t_p > 0.05 else "groups are NOT balanced (p <= 0.05)"

print(f"Control avg monthly_charges: £{control_charges_series.mean():.2f}")
print(f"Treatment avg monthly_charges: £{treatment_charges_series.mean():.2f}")
print(f"T-test: t-statistic={t_stat:.4f}, p-value={t_p:.4f}")
print(f"Result: {balance_result}")

Control avg monthly_charges: £78.92
Treatment avg monthly_charges: £79.79
T-test: t-statistic=-0.8024, p-value=0.4225
Result: groups are balanced (p > 0.05)


In [6]:
# Check 3: A/A test simulation — same conversion rate (35%) in both "groups", different
# random splits, repeated 1000 times, to confirm the testing methodology behaves as expected
aa_rng = np.random.RandomState(42)
false_positive_count = 0
n_simulations = 1000

for _ in range(n_simulations):
    aa_group_a = aa_rng.binomial(1, 0.35, 500)
    aa_group_b = aa_rng.binomial(1, 0.35, 500)
    _, aa_p_value = proportions_ztest([aa_group_a.sum(), aa_group_b.sum()], [500, 500])
    if aa_p_value < 0.05:
        false_positive_count += 1

false_positive_rate = false_positive_count / n_simulations * 100
print(f"False positive rate in A/A simulation: {false_positive_rate:.1f}% (expected ~5%)")

False positive rate in A/A simulation: 6.2% (expected ~5%)


## Step 3 — Statistical test

In [7]:
# Two-proportion z-test comparing control vs treatment churn rates
control_churn_count = ab_data.loc[ab_data["group"] == "control", "churned"].sum()
treatment_churn_count = ab_data.loc[ab_data["group"] == "treatment", "churned"].sum()

control_rate = control_churn_count / control_n
treatment_rate = treatment_churn_count / treatment_n

absolute_reduction = control_rate - treatment_rate
relative_reduction = absolute_reduction / control_rate

z_stat, p_value = proportions_ztest(
    [control_churn_count, treatment_churn_count], [control_n, treatment_n]
)
is_significant = p_value < 0.05

print(f"Control churn rate: {control_rate:.1%}")
print(f"Treatment churn rate: {treatment_rate:.1%}")
print(f"Absolute reduction: {absolute_reduction:.1%} percentage points")
print(f"Relative reduction: {relative_reduction:.1%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Result: {'SIGNIFICANT' if is_significant else 'NOT SIGNIFICANT'} at alpha=0.05")

Control churn rate: 34.6%
Treatment churn rate: 28.0%
Absolute reduction: 6.6% percentage points
Relative reduction: 19.1%
Z-statistic: 2.2504
P-value: 0.0244
Result: SIGNIFICANT at alpha=0.05


In [8]:
# 95% confidence interval for the difference in churn rates (normal approximation)
pooled_se = np.sqrt(
    control_rate * (1 - control_rate) / control_n + treatment_rate * (1 - treatment_rate) / treatment_n
)
ci_margin = 1.96 * pooled_se
diff_ci_lower = absolute_reduction - ci_margin
diff_ci_upper = absolute_reduction + ci_margin

print(f"We are 95% confident the true reduction is between {diff_ci_lower:.1%} and {diff_ci_upper:.1%}")

We are 95% confident the true reduction is between 0.9% and 12.3%


## Step 4 — Effect size

In [9]:
# Cohen's h for the difference between two proportions
cohens_h = proportion_effectsize(control_rate, treatment_rate)
abs_cohens_h = abs(cohens_h)

if abs_cohens_h < 0.2:
    effect_size_interpretation = "small"
elif abs_cohens_h < 0.5:
    effect_size_interpretation = "medium"
else:
    effect_size_interpretation = "large"

print(f"Cohen's h: {cohens_h:.4f}")
print(f"Effect size interpretation: {effect_size_interpretation} (small < 0.2, medium 0.2-0.5, large > 0.5)")

Cohen's h: 0.1425
Effect size interpretation: small (small < 0.2, medium 0.2-0.5, large > 0.5)


## Step 5 — Practical significance

In [10]:
# Scale the experiment's results up to the company's full high-risk population
total_high_risk_customers = 10000
avg_monthly_charge = ab_data["monthly_charges"].mean()
discount_rate = 0.10

baseline_churners = total_high_risk_customers * control_rate
treatment_churners = total_high_risk_customers * treatment_rate
customers_retained_by_discount = baseline_churners - treatment_churners

revenue_at_risk_control = baseline_churners * avg_monthly_charge
revenue_saved_by_treatment = customers_retained_by_discount * avg_monthly_charge

# Discount is offered to every customer who remains a customer under the treatment policy
customers_remaining_under_treatment = total_high_risk_customers - treatment_churners
cost_of_discount = customers_remaining_under_treatment * avg_monthly_charge * discount_rate

net_monthly_benefit = revenue_saved_by_treatment - cost_of_discount
annual_benefit = net_monthly_benefit * 12

print(f"Assumed population: {total_high_risk_customers:,} high-risk customers")
print(f"Average monthly charge: £{avg_monthly_charge:.2f}")
print(f"Monthly revenue at risk (control, no intervention): £{revenue_at_risk_control:,.2f}")
print(f"Monthly revenue saved by treatment (fewer churners): £{revenue_saved_by_treatment:,.2f}")
print(f"Monthly cost of discounts (10% off for {customers_remaining_under_treatment:,.0f} retained customers): £{cost_of_discount:,.2f}")
print(f"\nNet monthly benefit of rolling out discount: £{net_monthly_benefit:,.2f}")
print(f"Annual benefit: £{annual_benefit:,.2f}")

if net_monthly_benefit > 0:
    payback_months = cost_of_discount / net_monthly_benefit if net_monthly_benefit > 0 else None
    print("Break-even point: Immediate — net benefit is positive from month 1 (revenue saved exceeds discount cost every month).")
else:
    print("Break-even point: NEVER under a blanket rollout to all 10,000 customers — the discount "
          "cost of retaining the whole base outweighs the revenue saved from the customers who are "
          "actually kept from churning. A blanket rollout is not economically justified even though "
          "the churn reduction itself is real.")

Assumed population: 10,000 high-risk customers
Average monthly charge: £79.35
Monthly revenue at risk (control, no intervention): £274,560.65
Monthly revenue saved by treatment (fewer churners): £52,372.84
Monthly cost of discounts (10% off for 7,200 retained customers): £57,134.01

Net monthly benefit of rolling out discount: £-4,761.17
Annual benefit: £-57,134.01
Break-even point: NEVER under a blanket rollout to all 10,000 customers — the discount cost of retaining the whole base outweighs the revenue saved from the customers who are actually kept from churning. A blanket rollout is not economically justified even though the churn reduction itself is real.


## Step 6 — Segmented analysis

In [11]:
# Break results down by tenure segment and test each segment individually
segment_rows = []
for segment in tenure_segment_labels:
    segment_data = ab_data[ab_data["tenure_segment"] == segment]
    seg_control = segment_data[segment_data["group"] == "control"]["churned"]
    seg_treatment = segment_data[segment_data["group"] == "treatment"]["churned"]

    seg_control_rate = seg_control.mean()
    seg_treatment_rate = seg_treatment.mean()
    seg_reduction = seg_control_rate - seg_treatment_rate

    seg_z, seg_p = proportions_ztest(
        [seg_control.sum(), seg_treatment.sum()], [len(seg_control), len(seg_treatment)]
    )

    segment_rows.append({
        "Segment": segment,
        "Control Rate": seg_control_rate,
        "Treatment Rate": seg_treatment_rate,
        "Reduction": seg_reduction,
        "P-value": seg_p,
        "Significant?": "Yes" if seg_p < 0.05 else "No",
    })

segment_results = pd.DataFrame(segment_rows).sort_values("Reduction", ascending=False).reset_index(drop=True)

print("| Segment | Control Rate | Treatment Rate | Reduction | P-value | Significant? |")
print("|---------|--------------|-----------------|-----------|---------|---------------|")
for _, row in segment_results.iterrows():
    print(f"| {row['Segment']} | {row['Control Rate']:.1%} | {row['Treatment Rate']:.1%} | "
          f"{row['Reduction']:.1%} | {row['P-value']:.4f} | {row['Significant?']} |")

best_segment = segment_results.iloc[0]["Segment"]
print(f"\nSegment that benefits most from the discount (by absolute reduction): {best_segment}")
print("Note: with ~150-200 customers per group per segment, none of the individual segment "
      "tests may reach significance on their own even though the overall test does — this is "
      "expected given the smaller per-segment sample sizes, and should not be over-interpreted.")

| Segment | Control Rate | Treatment Rate | Reduction | P-value | Significant? |
|---------|--------------|-----------------|-----------|---------|---------------|
| Medium 12-36m | 34.8% | 25.5% | 9.3% | 0.0609 | No |
| Short <12m | 35.2% | 28.2% | 7.0% | 0.1341 | No |
| Long >36m | 33.3% | 30.7% | 2.7% | 0.6439 | No |

Segment that benefits most from the discount (by absolute reduction): Medium 12-36m
Note: with ~150-200 customers per group per segment, none of the individual segment tests may reach significance on their own even though the overall test does — this is expected given the smaller per-segment sample sizes, and should not be over-interpreted.


## Step 7 — Visualisations

### Chart 1 — Churn rate comparison

In [12]:
# Side-by-side bar chart with 95% CI error bars and an annotation of the difference
control_ci_low, control_ci_high = proportion_confint(control_churn_count, control_n, alpha=0.05, method="normal")
treatment_ci_low, treatment_ci_high = proportion_confint(treatment_churn_count, treatment_n, alpha=0.05, method="normal")

fig, ax = plt.subplots(figsize=(12, 6))
groups = ["Control", "Treatment"]
rates = [control_rate, treatment_rate]
errors = [
    [control_rate - control_ci_low, treatment_rate - treatment_ci_low],
    [control_ci_high - control_rate, treatment_ci_high - treatment_rate],
]
bars = ax.bar(groups, rates, yerr=errors, capsize=10, color=["#4C72B0", "#55A868"])
ax.set_title("Churn Rate: Control vs Treatment (with 95% CI)", fontsize=14, fontweight="bold")
ax.set_ylabel("Churn Rate")
for bar, rate in zip(bars, rates):
    ax.annotate(f"{rate:.1%}", (bar.get_x() + bar.get_width() / 2, rate), ha="center", va="bottom", fontsize=11)

# Leave headroom above the tallest error bar so the difference annotation doesn't collide with the title
top_of_bars = max(rates[i] + errors[1][i] for i in range(len(rates)))
ax.set_ylim(0, top_of_bars + 0.10)
ax.text(
    0.5, top_of_bars + 0.05, f"Difference: {absolute_reduction:.1%} pp (p={p_value:.4f})",
    transform=ax.get_yaxis_transform(), ha="center", fontsize=11, fontweight="bold",
)

save_chart(fig, project_path("outputs", "charts", "ab_test", "ab_churn_rate_comparison.png"))


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/ab_test/ab_churn_rate_comparison.png (50.2KB)


### Chart 2 — Segmented results

In [13]:
# Grouped bar chart: churn rate by tenure segment for control vs treatment
plot_data = ab_data.groupby(["tenure_segment", "group"])["churned"].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_data, x="tenure_segment", y="churned", hue="group",
            order=tenure_segment_labels, palette=["#4C72B0", "#55A868"], ax=ax)
ax.set_title("Churn Rate by Tenure Segment: Control vs Treatment", fontsize=14, fontweight="bold")
ax.set_xlabel("Tenure Segment")
ax.set_ylabel("Churn Rate")
ax.legend(title="Group")

save_chart(fig, project_path("outputs", "charts", "ab_test", "ab_segmented_results.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/ab_test/ab_segmented_results.png (56.0KB)


### Chart 3 — Confidence interval plot (forest plot)

In [14]:
# Forest-plot style chart showing the CI of the churn-rate reduction overall and per segment
forest_rows = [("Overall", absolute_reduction, diff_ci_lower, diff_ci_upper)]

for segment in tenure_segment_labels:
    segment_data = ab_data[ab_data["tenure_segment"] == segment]
    seg_control = segment_data[segment_data["group"] == "control"]["churned"]
    seg_treatment = segment_data[segment_data["group"] == "treatment"]["churned"]
    seg_control_rate = seg_control.mean()
    seg_treatment_rate = seg_treatment.mean()
    seg_reduction = seg_control_rate - seg_treatment_rate
    seg_se = np.sqrt(
        seg_control_rate * (1 - seg_control_rate) / len(seg_control)
        + seg_treatment_rate * (1 - seg_treatment_rate) / len(seg_treatment)
    )
    forest_rows.append((segment, seg_reduction, seg_reduction - 1.96 * seg_se, seg_reduction + 1.96 * seg_se))

forest_df = pd.DataFrame(forest_rows, columns=["Label", "Reduction", "CI_Low", "CI_High"])

fig, ax = plt.subplots(figsize=(12, 6))
y_positions = range(len(forest_df))
ax.errorbar(
    forest_df["Reduction"], y_positions,
    xerr=[forest_df["Reduction"] - forest_df["CI_Low"], forest_df["CI_High"] - forest_df["Reduction"]],
    fmt="o", capsize=6, color="#4C72B0", markersize=8,
)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_yticks(list(y_positions))
ax.set_yticklabels(forest_df["Label"])
ax.set_xlabel("Churn Rate Reduction (Control - Treatment), with 95% CI")
ax.set_title("Churn Rate Reduction — Overall and by Segment", fontsize=14, fontweight="bold")

save_chart(fig, project_path("outputs", "charts", "ab_test", "ab_confidence_intervals.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/ab_test/ab_confidence_intervals.png (59.0KB)


### Chart 4 — Business impact

In [15]:
# Bar chart summarising the monthly business impact of a blanket rollout
impact_labels = ["Revenue at Risk\n(Control)", "Revenue Saved\n(Treatment)", "Cost of Discounts", "Net Benefit"]
impact_values = [revenue_at_risk_control, revenue_saved_by_treatment, -cost_of_discount, net_monthly_benefit]
impact_colors = ["#C44E52", "#55A868", "#DD8452", "#4C72B0" if net_monthly_benefit >= 0 else "#C44E52"]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(impact_labels, impact_values, color=impact_colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly Business Impact of Discount Rollout (10,000 customers)", fontsize=14, fontweight="bold")
ax.set_ylabel("£ per month")
for bar, value in zip(bars, impact_values):
    va = "bottom" if value >= 0 else "top"
    ax.annotate(f"£{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value), ha="center", va=va, fontsize=10)

save_chart(fig, project_path("outputs", "charts", "ab_test", "ab_business_impact.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/ab_test/ab_business_impact.png (70.5KB)


## Step 8 — Decision memo

In [16]:
# Build the decision memo dynamically from the computed statistics above and render it as markdown
import datetime
today_str = datetime.date.today().strftime("%Y-%m-%d")

power_note = (
    "sufficient" if required_n_per_group <= 500
    else f"underpowered (would need ~{required_n_per_group:.0f} per group for 80% power at this effect size)"
)

if net_monthly_benefit > 0:
    recommendation_header = "GO"
    recommendation_text = (
        f"The treatment produces a statistically significant {absolute_reduction:.1%} pp reduction in churn "
        f"and, at scale, a positive net monthly benefit of £{net_monthly_benefit:,.0f}. Recommend a full rollout "
        f"of the 10% discount to all high-risk customers, while continuing to monitor the sample-size caveat below."
    )
else:
    recommendation_header = "CONDITIONAL GO"
    recommendation_text = (
        f"The treatment produces a statistically significant {absolute_reduction:.1%} pp reduction in churn — the "
        f"discount genuinely works. However, a blanket rollout to all {total_high_risk_customers:,} high-risk "
        f"customers is not profitable: the cost of discounting the whole retained base (£{cost_of_discount:,.0f}/month) "
        f"exceeds the revenue saved from the customers actually kept from churning (£{revenue_saved_by_treatment:,.0f}/month), "
        f"for a net monthly loss of £{abs(net_monthly_benefit):,.0f}. Recommend a **targeted rollout**: use the churn "
        f"model and risk scores from notebooks 3-5 to offer the discount only to customers flagged High Risk "
        f"(churn probability >= 0.70), where the discount cost is spent on the customers actually likely to leave, "
        f"rather than subsidising customers who would have stayed regardless."
    )

memo = f"""
---
# A/B Test Decision Memo
**Date:** {today_str}
**Test:** Retention Discount for High-Risk Customers
**Prepared by:** [Your Name]

## Executive Summary
A 10% retention discount reduced churn among high-risk customers from {control_rate:.1%} to {treatment_rate:.1%}, a statistically significant result (p={p_value:.4f}). Recommendation: **{recommendation_header}**, targeted rather than blanket, given the economics below.

## Test Results
| Metric | Value |
|--------|-------|
| Control churn rate | {control_rate:.1%} |
| Treatment churn rate | {treatment_rate:.1%} |
| Absolute reduction | {absolute_reduction:.1%} pp |
| Relative reduction | {relative_reduction:.1%} |
| P-value | {p_value:.4f} |
| Statistical significance | {"Yes" if is_significant else "No"} |
| Effect size (Cohen's h) | {cohens_h:.3f} ({effect_size_interpretation.capitalize()}) |
| 95% Confidence interval | [{diff_ci_lower:.1%}, {diff_ci_upper:.1%}] |

## Statistical Validity
- Sample Ratio Mismatch check: {srm_result} (both groups are exactly 500 customers, chi-square p={chi2_p:.4f})
- Pre-experiment balance on monthly_charges: {balance_result} (t-test p={t_p:.4f})
- A/A simulation false-positive rate: {false_positive_rate:.1f}% (expected ~5%), confirming the test procedure is well-calibrated
- Power analysis: the study is **{power_note}** to reliably detect a 5pp reduction at 80% power — treat the significant result with appropriate caution and prioritise a follow-up test with a larger sample

## Business Impact
- Monthly revenue at risk under status quo (control): £{revenue_at_risk_control:,.0f}
- Monthly revenue saved by the treatment: £{revenue_saved_by_treatment:,.0f}
- Monthly cost of discounting the retained base: £{cost_of_discount:,.0f}
- **Net monthly benefit of a blanket rollout: £{net_monthly_benefit:,.0f}** (Annual: £{annual_benefit:,.0f})

## Recommendation
**{recommendation_header}** — {recommendation_text}

## Next Steps
- Re-run the test with a larger sample (or longer duration) to reach the ~{required_n_per_group:.0f} customers per group needed for adequately powered results
- Use the churn model's risk scores (notebook 5) to restrict the discount to the High Risk segment only, and re-estimate the business case on that narrower, higher-value population
- Track long-term customer lifetime value (not just next-month churn) for discounted customers, since a discount could simply delay rather than prevent eventual churn

## Risks & Limitations
- The experiment data here is simulated for portfolio purposes; a real rollout would need to validate these effect sizes against actual customer behaviour
- The segmented analysis (Step 6) did not reach significance within any individual tenure segment due to smaller per-segment sample sizes, so segment-level targeting beyond the overall risk score should be treated as directional, not confirmed
---
"""

display(Markdown(memo))


---
# A/B Test Decision Memo
**Date:** 2026-07-03
**Test:** Retention Discount for High-Risk Customers
**Prepared by:** [Your Name]

## Executive Summary
A 10% retention discount reduced churn among high-risk customers from 34.6% to 28.0%, a statistically significant result (p=0.0244). Recommendation: **CONDITIONAL GO**, targeted rather than blanket, given the economics below.

## Test Results
| Metric | Value |
|--------|-------|
| Control churn rate | 34.6% |
| Treatment churn rate | 28.0% |
| Absolute reduction | 6.6% pp |
| Relative reduction | 19.1% |
| P-value | 0.0244 |
| Statistical significance | Yes |
| Effect size (Cohen's h) | 0.143 (Small) |
| 95% Confidence interval | [0.9%, 12.3%] |

## Statistical Validity
- Sample Ratio Mismatch check: PASS (both groups are exactly 500 customers, chi-square p=1.0000)
- Pre-experiment balance on monthly_charges: groups are balanced (p > 0.05) (t-test p=0.4225)
- A/A simulation false-positive rate: 6.2% (expected ~5%), confirming the test procedure is well-calibrated
- Power analysis: the study is **underpowered (would need ~1376 per group for 80% power at this effect size)** to reliably detect a 5pp reduction at 80% power — treat the significant result with appropriate caution and prioritise a follow-up test with a larger sample

## Business Impact
- Monthly revenue at risk under status quo (control): £274,561
- Monthly revenue saved by the treatment: £52,373
- Monthly cost of discounting the retained base: £57,134
- **Net monthly benefit of a blanket rollout: £-4,761** (Annual: £-57,134)

## Recommendation
**CONDITIONAL GO** — The treatment produces a statistically significant 6.6% pp reduction in churn — the discount genuinely works. However, a blanket rollout to all 10,000 high-risk customers is not profitable: the cost of discounting the whole retained base (£57,134/month) exceeds the revenue saved from the customers actually kept from churning (£52,373/month), for a net monthly loss of £4,761. Recommend a **targeted rollout**: use the churn model and risk scores from notebooks 3-5 to offer the discount only to customers flagged High Risk (churn probability >= 0.70), where the discount cost is spent on the customers actually likely to leave, rather than subsidising customers who would have stayed regardless.

## Next Steps
- Re-run the test with a larger sample (or longer duration) to reach the ~1376 customers per group needed for adequately powered results
- Use the churn model's risk scores (notebook 5) to restrict the discount to the High Risk segment only, and re-estimate the business case on that narrower, higher-value population
- Track long-term customer lifetime value (not just next-month churn) for discounted customers, since a discount could simply delay rather than prevent eventual churn

## Risks & Limitations
- The experiment data here is simulated for portfolio purposes; a real rollout would need to validate these effect sizes against actual customer behaviour
- The segmented analysis (Step 6) did not reach significance within any individual tenure segment due to smaller per-segment sample sizes, so segment-level targeting beyond the overall risk score should be treated as directional, not confirmed
---


## Save the Power BI export

In [17]:
# Save per-group churn rate summary with confidence intervals for Power BI
ab_test_results = pd.DataFrame([
    {
        "group": "control",
        "total_customers": control_n,
        "churned": control_churn_count,
        "churn_rate": control_rate,
        "lower_ci": control_ci_low,
        "upper_ci": control_ci_high,
    },
    {
        "group": "treatment",
        "total_customers": treatment_n,
        "churned": treatment_churn_count,
        "churn_rate": treatment_rate,
        "lower_ci": treatment_ci_low,
        "upper_ci": treatment_ci_high,
    },
])

save_dataframe(ab_test_results, project_path("data", "powerbi", "ab_test_results.csv"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/ab_test_results.csv (182.0B) — 2 rows, 6 columns
